# 🏆 HPO Champion Selection — Quick Guide

From raw Optuna runs to a tidy set of **per-strategy champions**, ready for validation.

---

## What this notebook does
1. **Load & score** HPO results  
2. **Filter robustly** (Quantile + MAD + optional Pareto)  
3. **Pick champions** (distinct physics strategies per well)  
4. **(Optional)** Visualize performance & hyperparameter importance  
5. **Export** a rich validation profile CSV

---

## One-screen flow

```text
Raw HPO Results → add_weighted_score()
               → run_distribution_filter()
                 ├─ quantile_thresholds
                 ├─ mad_guards
                 ├─ compose_predicates + apply gates
                 ├─ pareto_mark (optional)
                 └─ select_champions_by_strategy (Top-N distinct strategies per well)
               → Champions DataFrame → (optional) plots + validation_profile.csv


In [ ]:
"""
Master Dashboard for HPO Champion Selection (Notebook Entry Point)

This cell is the ONLY place you should edit to switch selection behavior.

Four explicit modes (no hybrids):
  1) LEGACY_WEIGHTED    -> legacy gates, selection_col='weighted_score'
  2) LEGACY_ROBUSTCOL   -> legacy gates, selection_col='robust_score'
  3) NEIGHBOR_TOP_PCT   -> neighborhood robust selector, pool_method='top_pct' (VAL-only), TEST audit-only
  4) NEIGHBOR_VAL_BAND  -> neighborhood robust selector, pool_method='val_band' (VAL-only), TEST audit-only

Optional new operational mode (SAFE):
  - ENABLE_PRI: runs PRI offline artifacts (leaderboard_enriched / pri_report / selection_policy)
    WITHOUT touching core selection, unless you switch MODE to PRI_POLICY.

Key semantics:
  - selector_mode controls WHICH selection path is executed (LEGACY vs NEIGHBOR).
  - scoring_strategy controls HOW scores are PRODUCED during aggregation (e.g., weighted_score / robust_score).
    These are intentionally independent.
"""
from __future__ import annotations

import sys
import json
import logging
from pathlib import Path
from typing import Any, Dict, List, Optional, Mapping

import pandas as pd


# ==============================================================================
# Project setup
# ==============================================================================
try:
    project_root = Path(__file__).parent.parent.parent  # script
except NameError:
    project_root = Path.cwd().parent.parent  # notebook

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from forecast_pipeline.io_utils import configure_logging
from common.experiment_context import ExperimentContext

from hpo.analysis import SelectionRunConfig, run_selection_pipeline

# You already moved these to hpo_runner (per your message)
from hpo.hpo_runner import (
    _print_mode_help,
    render_result,
    ModePreset,
)

# New: PRI offline orchestrator (does NOT touch core)
from hpo.pri_enrichment import run_pri_offline

configure_logging()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

try:
    from IPython.display import display
except ImportError:
    display = print


# ==============================================================================
# Presets (edit only this section)
# ==============================================================================

# --- Global toggles ---
GROUP = "HPO_156_Lag_100_Horizon_150"
# >>> Choose the mode here <<<
MODE = "LEGACY_WEIGHTED"  #LEGACY_WEIGHTED, LEGACY_ROBUSTCOL, NEIGHBOR_TOP_PCT
ARCH_SELECTION = "all"  # "all" | "seq2" | "darts" | "arps"

PLOT_SUMMARY_BARS = True
PLOT_ARCH_PERF = True
PLOT_HPARAM_IMPORTANCE = True
PLOT_CHAMPIONS = True

VALIDATION_RUN_NAME = "final_validation_of_champions"
VALIDATION_SEED = 42

SEQ2_ARCH_FILTER = "Seq2PIN"  # or None


# --- Scoring inputs used by aggregation ---
METRIC_WEIGHTS = {
    "val_smape_cum": 1.0,
    "val_smape_agg": 5.0,
}

LOWER_IS_BETTER = {
    "val_smape_cum": True,
    "val_smape_agg": True,
    "weighted_score": True,
    "robust_score": True,
}

# ==============================================================================
# PRI (offline) toggles — SAFE, does not touch core selection unless you opt-in
# ==============================================================================
ENABLE_PRI = False                 # para garantir que os artefatos PRI existam
PRI_FORCE_REBUILD = False         
ENABLE_PRI_ROUTING = False   # <<< liga roteamento (PRI_POLICY)
PRI_POLICY_PATH = None            # ou Path explícito, se quiser


# PRI grouping for report/policy
# SEMANTICS:
# - campaign does NOT exist in leaderboard -> campaign = GROUP (folder context)
# - dataset is NOT used (do not invent)
# - architecture is fixed per run (the notebook already isolates by arch)
PRI_GROUP_COLS = ["campaign", "well", "architecture"]

# Decision metric proxy (VAL-only).
# Keep as weighted_score when running LEGACY_WEIGHTED; you can swap to robust_score if needed.
PRI_VAL_METRIC_COL = "val_smape_agg"
PRI_K_GAP = 5
PRI_POOL_EPS = 0.02
PRI_RANK_METHOD = "spearman"  # informational (pri_enrichment uses spearman internally)

# Deterministic thresholds (audit-friendly)
PRI_THRESHOLDS = {
    "T_stab": 0.60,
    "T_gap": 0.02,
    "T_pool_eps": 0.02,
    "T_pool_max_frac": 0.30,
    "T_drift_p90": 3.0,
}

# Policy modes: what to recommend for stable vs unstable regimes
PRI_STABLE_MODE = "LEGACY_WEIGHTED"
PRI_UNSTABLE_MODE = "NEIGHBOR_TOP_PCT"

def _arch_list(selection: str) -> List[str]:
    s = (selection or "all").lower().strip()
    return [s] if s in {"seq2", "darts", "arps"} else ["seq2", "darts", "arps"]

def _arch_filter(arch: str) -> Optional[str]:
    return SEQ2_ARCH_FILTER if arch == "seq2" else None

def _arch_overrides(arch: str) -> Dict[str, Any]:
    return {"seq2": POSTHOC_OVERRIDES_SEQ2, "arps": POSTHOC_OVERRIDES_ARPS, "darts": POSTHOC_OVERRIDES_DARTS}.get(
        arch, POSTHOC_OVERRIDES_COMMON
    )

# --- Legacy posthoc overrides (used only by LEGACY_* modes) ---
POSTHOC_OVERRIDES_COMMON = dict(
    top_strategies_per_well=2,
    per_strategy_k=10,
    selection_strategy="best_of_the_best",
    apply_pareto=False,
    primary_quantile={"val_smape_agg": 0.6},
    mad_guard={"enabled": False, "alpha": 1, "metrics": ["val_smape_cum", "val_smape_agg"], "log": True, "side": "right"},
    relax_pool=False,
    valcum_gate={"q_low": 0.05, "q_high": 0.8},
)

POSTHOC_OVERRIDES_ARPS = {**POSTHOC_OVERRIDES_COMMON, "hpo_signature_cols": [
    "well", "campaign", "variant", "solver", "weighting", "loss", "lag_window", "horizon",
]}

POSTHOC_OVERRIDES_SEQ2 = {**POSTHOC_OVERRIDES_COMMON, "hpo_signature_cols": [
    "well", "campaign", "physics_strategy", "epochs", "batch_size", "learning_rate", "data_sample",
]}

POSTHOC_OVERRIDES_DARTS = {**POSTHOC_OVERRIDES_COMMON, "hpo_signature_cols": [
    "well", "campaign", "profile", "physics_strategy", "n_epochs", "batch_size", "learning_rate",
]}


# --- Neighborhood overrides (used only by NEIGHBOR_* modes) ---
DEFAULT_NEIGHBORHOOD = dict(
    pool_cfg=dict(top_pct=0.1, drop=0.05, take=0.40, min_candidates=20),
    robust_cfg=dict(k=20, min_strat=25, alpha=0.65, beta=0.03, gamma=0.35, luck_q=0.25, w_lr=2.0, w_ep=0.25, w_bs=0.50),
    group_cols=["dataset", "well", "architecture"],
)

PRESETS: Dict[str, ModePreset] = {
    "LEGACY_WEIGHTED": ModePreset(
        name="LEGACY_WEIGHTED",
        selector_mode="LEGACY_WEIGHTED",
        scoring_strategy="weighted_score",
        metric_to_optimize="weighted_score",
        neighborhood_overrides={},
        posthoc_overrides={},
        help="Legacy gates + dedup + champions chosen by weighted_score (lower is better). Pool is locked to survivors.",
    ),
    "LEGACY_ROBUSTCOL": ModePreset(
        name="LEGACY_ROBUSTCOL",
        selector_mode="LEGACY_ROBUSTCOL",
        scoring_strategy="robust_score",
        metric_to_optimize="robust_score",
        neighborhood_overrides={},
        posthoc_overrides={},
        help="Legacy gates + dedup + champions chosen by robust_score (lower is better). Pool is locked to survivors.",
    ),
    "NEIGHBOR_TOP_PCT": ModePreset(
        name="NEIGHBOR_TOP_PCT",
        selector_mode="NEIGHBOR_TOP_PCT",
        scoring_strategy="robust_score",
        metric_to_optimize="robust_score",
        neighborhood_overrides={**DEFAULT_NEIGHBORHOOD, "pool_method": "top_pct"},
        posthoc_overrides={},
        help="Neighborhood robust selector: pool = top_pct (VAL-only), kNN local scoring in HP-space, TEST is audit-only.",
    ),
    "NEIGHBOR_VAL_BAND": ModePreset(
        name="NEIGHBOR_VAL_BAND",
        selector_mode="NEIGHBOR_VAL_BAND",
        scoring_strategy="robust_score",
        metric_to_optimize="robust_score",
        neighborhood_overrides={**DEFAULT_NEIGHBORHOOD, "pool_method": "val_band"},
        posthoc_overrides={},
        help="Neighborhood robust selector: pool = val_band (VAL-only), kNN local scoring in HP-space, TEST is audit-only.",
    ),
}



def _resolve_master_leaderboard_path(ctx: ExperimentContext) -> Optional[Path]:
    """
    Prefere sempre o master, pois é o único que representa o agregado completo (ex.: 2304 trials).
    Por padrão, ele está em: reports/<arch>/hpo_master_leaderboard.csv
    """
    p = Path(ctx.reports_dir_arch) / "hpo_master_leaderboard.csv"
    return p if p.exists() else None


def _run_pri_for_arch(ctx: ExperimentContext, *, arch: str, preset: ModePreset) -> Optional[Dict[str, Any]]:
    """
    PRI é SIDE-CAR: nunca deve derrubar a seleção.
    Estratégia:
      1) Se existir master leaderboard em reports_dir_arch: usa ele (leaderboard_path=...),
         NÃO injeta campaign_name (master já tem coluna 'campaign' correta).
      2) Caso não exista master: roda modo legado (descobre leaderboard via results_dir),
         injeta campaign_name=GROUP (pois cycle leaderboard tipicamente não tem 'campaign').

    Também faz fallback automático de val_metric_col.
    """
    if not ENABLE_PRI:
        return None

    master_lb = _resolve_master_leaderboard_path(ctx)
    use_master = master_lb is not None

    # Ordem de fallback (tente o que o usuário pediu, depois coisas que provavelmente existem)
    metric_candidates = [
        str(PRI_VAL_METRIC_COL).strip(),
        str(getattr(preset, "metric_to_optimize", "")).strip(),
        str(getattr(preset, "scoring_strategy", "")).strip(),
        "val_smape_agg",
        "val_smape_cum",
    ]
    # limpa vazios e duplicados preservando ordem
    seen = set()
    metric_candidates = [m for m in metric_candidates if m and not (m in seen or seen.add(m))]

    last_err: Optional[Exception] = None

    for mcol in metric_candidates:
        try:
            out = run_pri_offline(
                results_dir=ctx.results_dir,
                reports_dir=ctx.reports_dir_arch,
                group_cols=list(PRI_GROUP_COLS),
                val_metric_col=str(mcol),
                k_gap=int(PRI_K_GAP),
                pool_eps=float(PRI_POOL_EPS),
                rank_method=str(PRI_RANK_METHOD),
                thresholds=dict(PRI_THRESHOLDS),
                stable_mode=str(PRI_STABLE_MODE),
                unstable_mode=str(PRI_UNSTABLE_MODE),
                force=bool(PRI_FORCE_REBUILD),
                enable_sanity_check=True,

                # ✅ master se existir
                leaderboard_path=master_lb if use_master else None,

                # ✅ semântica correta:
                # - master já tem campaign real -> NÃO sobrescreve
                # - cycle não tem campaign -> injeta GROUP
                campaign_name=None if use_master else str(GROUP),

                # você pode fixar architecture para manter grouping consistente
                architecture_value=str(arch),
            )

            print("\n--- PRI offline (VAL-only) ---")
            print(f"source : {'master' if use_master else 'cycle'}")
            print(f"leaderboard_in: {out.get('meta', {}).get('leaderboard_in', out.get('leaderboard_in', None)) or (str(master_lb) if use_master else 'auto-discovery')}")
            print(f"val_metric_col: {mcol}")
            if out.get("skipped"):
                print(f"SKIP: {out.get('reason')} | results_dir={out.get('results_dir', str(ctx.results_dir))}")
                return out

            print(f"enriched : {out.get('leaderboard_enriched_path')}")
            print(f"pri_report: {out.get('pri_report_path')}")
            print(f"policy  : {out.get('selection_policy_path')}")
            cov = out.get("coverage")
            if isinstance(cov, (int, float)):
                print(f"coverage: {cov:.4f} | missing_json={out.get('missing_json')} | missing_val_series={out.get('missing_val_series')}")
            else:
                print(f"coverage: {cov} | missing_json={out.get('missing_json')} | missing_val_series={out.get('missing_val_series')}")
            return out

        except FileNotFoundError:
            # sidecar: não mata o run
            print("\n--- PRI offline (VAL-only) ---")
            print(f"SKIP: leaderboard_not_found | results_dir={ctx.results_dir}")
            return {"skipped": True, "reason": "leaderboard_not_found", "results_dir": str(ctx.results_dir)}

        except ValueError as e:
            # Caso clássico: "Missing required PRI columns=['weighted_score']"
            msg = str(e)
            last_err = e
            if "Missing required PRI columns" in msg:
                # tenta próximo candidato
                continue
            logging.error("⚠️ PRI offline ValueError (ignored): %s", msg, exc_info=True)
            return {"skipped": True, "reason": f"value_error:{type(e).__name__}", "details": msg}

        except Exception as e:
            last_err = e
            logging.error("⚠️ PRI offline failed (ignored): %s: %s", type(e).__name__, e, exc_info=True)
            return {"skipped": True, "reason": f"pri_failed:{type(e).__name__}", "details": str(e)}

    print("\n--- PRI offline (VAL-only) ---")
    print(f"SKIP: no_valid_val_metric_col_found | tried={metric_candidates} | last_err={last_err}")
    return {"skipped": True, "reason": "no_valid_val_metric_col_found", "tried": metric_candidates, "details": str(last_err) if last_err else None}




def run_one_architecture(arch: str, preset: ModePreset) -> Dict[str, Any]:
    ctx = ExperimentContext(group=GROUP, arch=arch)

    # 0) Optional: PRI offline (sidecar)
    _run_pri_for_arch(ctx, arch=arch, preset=preset)

    # 1) Legacy/Neighbor selection as-is (unchanged)
    posthoc_over = dict(_arch_overrides(arch))
    posthoc_over.update(preset.posthoc_overrides or {})

    cfg = SelectionRunConfig(
        results_dir=ctx.results_dir,
        reports_dir=ctx.reports_dir_arch,
        metric_weights=METRIC_WEIGHTS,
        lower_is_better=LOWER_IS_BETTER,
        scoring_strategy=preset.scoring_strategy,
        metric_to_optimize=preset.metric_to_optimize,
        selector_mode=preset.selector_mode,
        neighborhood_overrides=dict(preset.neighborhood_overrides or {}),
        posthoc_overrides=posthoc_over,
        arch_filter=_arch_filter(arch),
        plot_architecture_performance=PLOT_ARCH_PERF,
        plot_hparam_importance_per_well=PLOT_HPARAM_IMPORTANCE,
        plot_champions_per_well=PLOT_CHAMPIONS,
        validation_run_name=VALIDATION_RUN_NAME,
        validation_seed=VALIDATION_SEED,
        plot_summary_bars=PLOT_SUMMARY_BARS,

        # >>> NEW: PRI routing knobs <<<
        enable_pri_routing=bool(ENABLE_PRI_ROUTING),
        pri_policy_path=str(PRI_POLICY_PATH) if PRI_POLICY_PATH else None,
    )


    return run_selection_pipeline(cfg)


# ==============================================================================
# Execute
# ==============================================================================
preset = PRESETS.get(str(MODE).upper().strip())
if preset is None:
    raise ValueError(f"Unknown MODE='{MODE}'. Available={list(PRESETS.keys())}")

_print_mode_help(preset)

for arch in _arch_list(ARCH_SELECTION):
    print("\n" + "-" * 90)
    print(f"Running ARCH={arch.upper()} | GROUP={GROUP} | MODE={preset.name} | ENABLE_PRI={ENABLE_PRI}")
    print("-" * 90)
    try:
        res = run_one_architecture(arch, preset)
        render_result(res, preset)
    except Exception as e:
        logging.error("❌ %s RUN FAILED: %s: %s", arch.upper(), type(e).__name__, e, exc_info=True)


In [ ]:
# ============================================================
# HPO Champion Selection — Benchmark Runner (single cell) [COMPACT]
# - Runs GROUPS × all PRESETS × architectures (per ARCH_SELECTION_BENCH)
# - Reuses SelectionRunConfig + run_selection_pipeline (no logic changes)
# - Captures result["summary"] (TEST audit-only) and derives quick-audit stats
# - Produces:
#   (A) Winners per group (1 row per GROUP)
#   (B) Compact Group × Mode scoreboard (only key metrics)
#   (C) Global ranking by mode across groups (compact)
# - Keeps RAW[(group, mode, arch)] for drilldown
# ============================================================

from __future__ import annotations

import logging
import math
from typing import Any, Dict, List, Mapping, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# --------------------------
# USER INPUTS (edit here)
# --------------------------
GROUPS = [
     "HPO_150_Lag_100_Horizon_150",
    # "HPO_72_Lag_100_Horizon_150",
    # "HPO_64_Lag_100_Horizon_150",
    # "HPO_156_Lag_100_Horizon_150",
    # "HPO_60_Lag_100_Horizon_150",
    # "HPO_153_Lag_100_Horizon_150"
    
]


ARCH_SELECTION_BENCH = "all"  # "all" | "seq2" | "darts" | "arps"
DISABLE_PLOTS_FOR_BENCH = True

# Winner criteria (lexicographic, deterministic)
# 1) min regret_median, 2) min ratio_median, 3) max spearman_median, 4) min regret_p90, 5) mode name tie-break
WINNER_TIEBREAK_MODE_NAME = True


# =============================================================================
# Helpers (small, local)
# =============================================================================
def _arch_list_local(selection: str) -> List[str]:
    s = (selection or "all").lower().strip()
    return [s] if s in {"seq2", "darts", "arps"} else ["seq2", "darts", "arps"]

def _nanquantile(x: np.ndarray, q: float) -> float:
    x = x[np.isfinite(x)]
    if x.size == 0:
        return float("nan")
    return float(np.quantile(x, q))

def _coalesce_xy_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Coalesce columns with _x/_y suffixes into a base column.
    Prefer _y, fallback to _x, fallback to base.
    """
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return df

    cols = list(df.columns)
    bases: Dict[str, Dict[str, str]] = {}
    for c in cols:
        if c.endswith("_x"):
            bases.setdefault(c[:-2], {})["x"] = c
        elif c.endswith("_y"):
            bases.setdefault(c[:-2], {})["y"] = c

    out = df.copy()
    for base, d in bases.items():
        y = d.get("y")
        x = d.get("x")
        if y is None and x is None:
            continue

        base_series = out[base] if base in out.columns else pd.Series([np.nan] * len(out), index=out.index)

        if y is not None and y in out.columns:
            base_series = base_series.where(base_series.notna(), out[y])
        if x is not None and x in out.columns:
            base_series = base_series.where(base_series.notna(), out[x])

        out[base] = base_series

    return out

def _audit_stats(summary_df: Optional[pd.DataFrame]) -> Dict[str, float]:
    """
    Compute quick-audit stats from result["summary"] (TEST is audit-only).
    """
    if summary_df is None or not isinstance(summary_df, pd.DataFrame) or summary_df.empty:
        return {
            "n_rows": 0,
            "regret_median": np.nan, "regret_p90": np.nan,
            "ratio_median": np.nan, "ratio_p90": np.nan,
            "spearman_median": np.nan,
        }

    df = _coalesce_xy_columns(summary_df)

    regret_col = "regret_test" if "regret_test" in df.columns else None
    ratio_col = "ratio_test" if "ratio_test" in df.columns else None
    spear_col = "val_test_spearman" if "val_test_spearman" in df.columns else ("spearman" if "spearman" in df.columns else None)

    regret = pd.to_numeric(df[regret_col], errors="coerce").to_numpy(dtype=float) if regret_col else np.array([], dtype=float)
    ratio  = pd.to_numeric(df[ratio_col], errors="coerce").to_numpy(dtype=float) if ratio_col else np.array([], dtype=float)
    spear  = pd.to_numeric(df[spear_col], errors="coerce").to_numpy(dtype=float) if spear_col else np.array([], dtype=float)

    return {
        "n_rows": int(len(df)),
        "regret_median": _nanquantile(regret, 0.50) if regret.size else np.nan,
        "regret_p90": _nanquantile(regret, 0.90) if regret.size else np.nan,
        "ratio_median": _nanquantile(ratio, 0.50) if ratio.size else np.nan,
        "ratio_p90": _nanquantile(ratio, 0.90) if ratio.size else np.nan,
        "spearman_median": _nanquantile(spear, 0.50) if spear.size else np.nan,
    }

def _winner_rank_key(row: Mapping[str, Any]) -> Tuple[float, float, float, float, str]:
    """
    Deterministic lexicographic rank key:
      1) min regret_median
      2) min ratio_median
      3) max spearman_median -> we encode as -spearman_median
      4) min regret_p90
      5) tie-break by mode name (optional)
    """
    def _f(v: Any, default: float) -> float:
        try:
            v = float(v)
            return v if math.isfinite(v) else default
        except Exception:
            return default

    r50 = _f(row.get("regret_median"), default=1e18)
    rr50 = _f(row.get("ratio_median"), default=1e18)
    s50 = _f(row.get("spearman_median"), default=-1e18)
    r90 = _f(row.get("regret_p90"), default=1e18)
    mode = str(row.get("mode", "")) if WINNER_TIEBREAK_MODE_NAME else ""
    return (r50, rr50, -s50, r90, mode)

def _collapse_group_mode_minimal(g: pd.DataFrame) -> Dict[str, Any]:
    """
    Collapse arch-level rows -> (group, mode) row with ONLY presentation metrics.
    Uses conservative worst-case across arches:
      regret_median = max across arches
      ratio_median  = max across arches
      spearman_med  = min across arches
      regret_p90    = max across arches
      ratio_p90     = max across arches
    """
    group = str(g["group"].iloc[0])
    mode = str(g["mode"].iloc[0])

    status_counts = g["status"].value_counts(dropna=False).to_dict()
    n_ok = int(status_counts.get("ok", 0))
    n_error = int(status_counts.get("error", 0))

    # primary status
    if n_error > 0:
        status = "error"
    elif n_ok == 0:
        status = "empty"
    else:
        status = "ok"

    def _num(col: str) -> pd.Series:
        return pd.to_numeric(g[col], errors="coerce")

    return {
        "group": group,
        "mode": mode,
        "status": status,
        "regret_median": float(_num("regret_median").max(skipna=True)),
        "regret_p90": float(_num("regret_p90").max(skipna=True)),
        "ratio_median": float(_num("ratio_median").max(skipna=True)),
        "ratio_p90": float(_num("ratio_p90").max(skipna=True)),
        "spearman_median": float(_num("spearman_median").min(skipna=True)),
    }

def _style_compact(df: pd.DataFrame, *, winner_mask_col: Optional[str] = None) -> "pd.io.formats.style.Styler":
    """
    Compact, readable styling:
    - Larger fonts, padding, subtle zebra rows
    - Highlight winner rows if winner_mask_col is provided
    - Gradients: regret/ratio lower=better, spearman higher=better
    """
    if df is None or df.empty:
        return df.style

    low_good = [c for c in ["regret_median", "regret_p90", "ratio_median", "ratio_p90"] if c in df.columns]
    high_good = [c for c in ["spearman_median"] if c in df.columns]

    def _zebra(i: int) -> str:
        return "background-color: rgba(0,0,0,0.03);" if (i % 2 == 1) else ""

    def _row_style(row: pd.Series) -> List[str]:
        css = []
        is_winner = bool(row.get(winner_mask_col, False)) if winner_mask_col else False
        status = str(row.get("status", "")).lower()
        for j, _ in enumerate(row.index):
            base = _zebra(int(row.name)) if isinstance(row.name, (int, np.integer)) else ""
            if status == "error":
                css.append(base + "background-color: rgba(255, 0, 0, 0.08);")
            elif status == "empty":
                css.append(base + "background-color: rgba(0, 0, 0, 0.06);")
            elif is_winner:
                css.append(base + "background-color: rgba(0, 180, 0, 0.12); font-weight: 700;")
            else:
                css.append(base)
        return css

    styler = df.style.apply(_row_style, axis=1)
    styler = styler.set_table_styles([
        {"selector": "th", "props": [("font-size", "14px"), ("text-align", "left"), ("padding", "8px 10px")]},
        {"selector": "td", "props": [("font-size", "14px"), ("padding", "6px 10px")]},
        {"selector": "table", "props": [("width", "100%")]},
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "16px"), ("font-weight", "700")]}
    ])

    # Numeric formatting
    fmt = {c: "{:.3g}" for c in (low_good + high_good)}
    styler = styler.format(fmt, na_rep="—")

    # Gradients
    if low_good:
        styler = styler.background_gradient(subset=low_good, axis=0)
    if high_good:
        # Custom subtle alpha based on rank (higher better)
        def _grad_high(col: pd.Series) -> List[str]:
            x = pd.to_numeric(col, errors="coerce")
            r = x.rank(pct=True, ascending=True)  # higher -> higher pct
            return [f"background-color: rgba(0, 140, 255, {0.22 * (p if pd.notna(p) else 0)})" for p in r]
        for c in high_good:
            styler = styler.apply(lambda s, col=c: _grad_high(df[col]), axis=0, subset=[c])

    return styler

def _best_and_runnerup_per_group(df_scoreboard_ok: pd.DataFrame) -> pd.DataFrame:
    """
    1 row per group:
      group, winner_mode, runner_up_mode,
      winner_* metrics (regret_median/ratio_median/spearman_median/regret_p90)
    """
    rows = []
    for group, g in df_scoreboard_ok.groupby("group", sort=False):
        g2 = g.copy()
        g2["__rk__"] = g2.apply(lambda r: _winner_rank_key(r), axis=1)
        g2 = g2.sort_values(["__rk__", "mode"], kind="mergesort").reset_index(drop=True)

        w = g2.iloc[0]
        ru = g2.iloc[1] if len(g2) > 1 else None

        rows.append({
            "group": group,
            "winner_mode": str(w["mode"]),
            "runner_up_mode": str(ru["mode"]) if ru is not None else "—",
            "regret_median": float(w["regret_median"]) if pd.notna(w["regret_median"]) else np.nan,
            "ratio_median": float(w["ratio_median"]) if pd.notna(w["ratio_median"]) else np.nan,
            "spearman_median": float(w["spearman_median"]) if pd.notna(w["spearman_median"]) else np.nan,
            "regret_p90": float(w["regret_p90"]) if pd.notna(w["regret_p90"]) else np.nan,
        })
    return pd.DataFrame(rows)

def _global_mode_ranking(df_ok: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate across groups (only ok rows):
      - global medians + p90 medians across groups (same semantics as before)
      - plus worst-group regret_median (max) for safety
    """
    def med(col: str) -> float:
        return float(pd.to_numeric(df_ok[col], errors="coerce").median(skipna=True))

    rows = []
    for mode, g in df_ok.groupby("mode", sort=False):
        rows.append({
            "mode": mode,
            "ok_groups": int(g["group"].nunique()),
            "total_groups": int(len(GROUPS)),
            "regret_median_global": float(pd.to_numeric(g["regret_median"], errors="coerce").median(skipna=True)),
            "ratio_median_global": float(pd.to_numeric(g["ratio_median"], errors="coerce").median(skipna=True)),
            "spearman_median_global": float(pd.to_numeric(g["spearman_median"], errors="coerce").median(skipna=True)),
            "regret_p90_global": float(pd.to_numeric(g["regret_p90"], errors="coerce").median(skipna=True)),
            "ratio_p90_global": float(pd.to_numeric(g["ratio_p90"], errors="coerce").median(skipna=True)),
            "regret_median_worst_group": float(pd.to_numeric(g["regret_median"], errors="coerce").max(skipna=True)),
        })
    out = pd.DataFrame(rows)

    def _global_rank_key(r: Mapping[str, Any]) -> Tuple[float, float, float, float, str]:
        rr = {
            "regret_median": r.get("regret_median_global"),
            "ratio_median": r.get("ratio_median_global"),
            "spearman_median": r.get("spearman_median_global"),
            "regret_p90": r.get("regret_p90_global"),
            "mode": r.get("mode"),
        }
        return _winner_rank_key(rr)

    out["__rk__"] = out.apply(lambda r: _global_rank_key(r), axis=1)
    out = out.sort_values(["__rk__", "mode"], kind="mergesort").reset_index(drop=True)
    out.insert(0, "global_rank", np.arange(1, len(out) + 1))
    out = out.drop(columns="__rk__", errors="ignore")
    return out


# =============================================================================
# Benchmark runner
# =============================================================================
# Expect these to exist in your notebook context (from your dashboard cell):
#   PRESETS, ModePreset
#   METRIC_WEIGHTS, LOWER_IS_BETTER
#   _arch_filter, _arch_overrides
#   VALIDATION_RUN_NAME, VALIDATION_SEED
#   PLOT_* toggles
#   ExperimentContext, SelectionRunConfig, run_selection_pipeline

arches = _arch_list_local(ARCH_SELECTION_BENCH)
modes = list(PRESETS.keys())

if DISABLE_PLOTS_FOR_BENCH:
    PLOT_SUMMARY_BARS = False
    PLOT_ARCH_PERF = False
    PLOT_HPARAM_IMPORTANCE = False
    PLOT_CHAMPIONS = False

RAW: Dict[Tuple[str, str, str], Dict[str, Any]] = {}
rows_long: List[Dict[str, Any]] = []

for group in GROUPS:
    for mode_name in modes:
        preset = PRESETS[mode_name]
        for arch in arches:
            try:
                ctx = ExperimentContext(group=group, arch=arch)

                posthoc_over = dict(_arch_overrides(arch))
                posthoc_over.update(preset.posthoc_overrides or {})

                cfg = SelectionRunConfig(
                    results_dir=ctx.results_dir,
                    reports_dir=ctx.reports_dir_arch,
                    metric_weights=METRIC_WEIGHTS,
                    lower_is_better=LOWER_IS_BETTER,
                    scoring_strategy=preset.scoring_strategy,
                    metric_to_optimize=preset.metric_to_optimize,
                    selector_mode=preset.selector_mode,
                    neighborhood_overrides=dict(preset.neighborhood_overrides or {}),
                    posthoc_overrides=posthoc_over,
                    arch_filter=_arch_filter(arch),
                    plot_architecture_performance=PLOT_ARCH_PERF,
                    plot_hparam_importance_per_well=PLOT_HPARAM_IMPORTANCE,
                    plot_champions_per_well=PLOT_CHAMPIONS,
                    validation_run_name=VALIDATION_RUN_NAME,
                    validation_seed=VALIDATION_SEED,
                    plot_summary_bars=PLOT_SUMMARY_BARS,
                )

                result = run_selection_pipeline(cfg)
                meta = result.get("meta", {}) or {}
                summary = result.get("summary")

                status = "ok" if isinstance(summary, pd.DataFrame) and not summary.empty else "empty"
                stats = _audit_stats(summary)

                rows_long.append({
                    "group": group,
                    "mode": mode_name,
                    "arch": arch,
                    "status": status,
                    **stats,
                })

                RAW[(group, mode_name, arch)] = {"meta": meta, "summary": summary, "result": result}

            except Exception as e:
                logging.error("[benchmark] FAILED | group=%s mode=%s arch=%s | %s: %s",
                              group, mode_name, arch, type(e).__name__, e, exc_info=True)
                rows_long.append({
                    "group": group, "mode": mode_name, "arch": arch, "status": "error",
                    "n_rows": 0,
                    "regret_median": np.nan, "regret_p90": np.nan,
                    "ratio_median": np.nan, "ratio_p90": np.nan,
                    "spearman_median": np.nan,
                })

df_long = pd.DataFrame(rows_long)

# (group, mode) scoreboard (minimal columns)
df_scoreboard = (
    df_long.groupby(["group", "mode"], as_index=False, sort=False)
    .apply(lambda x: pd.Series(_collapse_group_mode_minimal(x)))
    .reset_index(drop=True)
)

# Mark winners per group
df_scoreboard["__rk__"] = df_scoreboard.apply(lambda r: _winner_rank_key(r), axis=1)
df_scoreboard["is_winner_group"] = False

for group in GROUPS:
    m = (df_scoreboard["group"] == group) & (df_scoreboard["status"] == "ok")
    if not m.any():
        continue
    sub = df_scoreboard.loc[m].copy()
    sub = sub.sort_values(["__rk__", "mode"], kind="mergesort")
    df_scoreboard.loc[sub.index[0], "is_winner_group"] = True

df_scoreboard = df_scoreboard.sort_values(
    ["group", "is_winner_group", "__rk__", "mode"],
    ascending=[True, False, True, True],
    kind="mergesort"
).reset_index(drop=True)

# Winners table (1 row per group)
df_ok = df_scoreboard[df_scoreboard["status"] == "ok"].copy()
df_winners = _best_and_runnerup_per_group(df_ok)

# Global ranking (compact)
df_global = _global_mode_ranking(df_ok)

# Clean display columns (per your request)
score_cols = ["group", "mode", "status", "regret_median", "regret_p90", "ratio_median", "ratio_p90", "spearman_median", "is_winner_group"]
score_cols = [c for c in score_cols if c in df_scoreboard.columns]
df_score_compact = df_scoreboard.loc[:, score_cols].copy()

# --------------------------
# Display (compact + rich)
# --------------------------
print("\n(A) Winners per GROUP (1 row per group):")
display(_style_compact(df_winners, winner_mask_col=None).set_caption("Winners per Group"))

print("\n(B) Group × Mode scoreboard (compact; winners highlighted):")
display(_style_compact(df_score_compact, winner_mask_col="is_winner_group").set_caption("Group × Mode Scoreboard"))

print("\n(C) Global ranking by MODE (compact):")
global_cols = [
    "global_rank", "mode", "ok_groups", "total_groups",
    "regret_median_global", "regret_p90_global",
    "ratio_median_global", "ratio_p90_global",
    "spearman_median_global",
    "regret_median_worst_group",
]
global_cols = [c for c in global_cols if c in df_global.columns]
display(_style_compact(df_global.loc[:, global_cols].copy(), winner_mask_col=None).set_caption("Global Ranking by Mode"))

# --------------------------
# Optional drilldown
# --------------------------
print("\nDrilldown tips:")
print("- RAW[(group, mode, arch)]['summary'] gives the per-campaign regret table (TEST audit-only).")
print("- Example: display(RAW[('HPO_280_Lag_100_Horizon_150','NEIGHBOR_VAL_BAND','seq2')]['summary'])")
